# Pipeline 產出物檢視

手動檢視 pipeline 執行後的產出物（parquet、pickle、json），用於除錯和驗證資料品質。

In [12]:
"""設定環境：載入配置與建立 DataCatalog"""
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()))

from recsys_tfb.core.config import ConfigLoader
from recsys_tfb.core.catalog import DataCatalog

config = ConfigLoader("conf", env="local")
catalog = DataCatalog(config.get_catalog_config())

print("已註冊的資料集：")
for name in catalog.list():
    status = "存在" if catalog.exists(name) else "不存在"
    print(f"  - {name} ({status})")

已註冊的資料集：
  - feature_table (存在)
  - label_table (存在)
  - model (不存在)
  - preprocessor (存在)
  - category_mappings (存在)


In [13]:
"""檢視 feature_table（Parquet）"""
if catalog.exists("feature_table"):
    df = catalog.load("feature_table")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))
    display(df.describe())
else:
    print("feature_table 不存在，請先執行 dataset building pipeline。")

Shape: (600, 8)

Dtypes:
snap_date            datetime64[ns]
cust_id                      object
total_aum                   float64
fund_aum                    float64
in_amt_sum_l1m              float64
out_amt_sum_l1m             float64
in_amt_ratio_l1m            float64
out_amt_ratio_l1m           float64
dtype: object

Null 統計:
snap_date            0
cust_id              0
total_aum            0
fund_aum             0
in_amt_sum_l1m       0
out_amt_sum_l1m      0
in_amt_ratio_l1m     0
out_amt_ratio_l1m    0
dtype: int64


,snap_date,cust_id,total_aum,fund_aum,in_amt_sum_l1m,out_amt_sum_l1m,in_amt_ratio_l1m,out_amt_ratio_l1m
0,2024-01-31,C000001,1202104.30,300971.57,40446.55,168057.34,0.033646,0.139803
1,2024-01-31,C000002,1168094.83,84042.96,13239.03,31761.59,0.011334,0.027191
2,2024-01-31,C000003,1192380.50,8308.68,21470.19,58477.36,0.018006,0.049043
3,2024-01-31,C000004,139897.14,16064.11,70948.52,35576.96,0.507148,0.254308
4,2024-01-31,C000005,43218.70,2848.59,878.64,29483.60,0.020330,0.682195
5,2024-01-31,C000006,726330.26,246102.00,88826.99,23012.49,0.122296,0.031683
6,2024-01-31,C000007,704980.35,42944.76,1057.21,975.01,0.001500,0.001383
7,2024-01-31,C000008,1562147.98,395481.14,47877.42,157156.66,0.030648,0.100603
8,2024-01-31,C000009,39647.10,13762.75,2952.64,41669.29,0.074473,1.051005
9,2024-01-31,C000010,523280.42,152043.47,16002.50,110398.02,0.030581,0.210973


,total_aum,fund_aum,in_amt_sum_l1m,out_amt_sum_l1m,in_amt_ratio_l1m,out_amt_ratio_l1m
count,6.000000e+02,6.000000e+02,600.000000,600.000000,600.000000,600.000000
mean,5.187228e+05,1.251505e+05,50096.957517,42286.227150,0.440432,0.377019
std,5.342478e+05,1.606115e+05,51677.400056,42658.819182,1.937659,1.639452
min,6.904800e+02,8.389000e+01,24.210000,44.190000,0.000103,0.000196
25%,1.597485e+05,2.283696e+04,14033.107500,11118.692500,0.031773,0.026942
50%,3.615551e+05,7.010671e+04,32513.165000,28804.505000,0.096098,0.079920
75%,6.962938e+05,1.643068e+05,69245.105000,57599.990000,0.304552,0.219749
max,3.807333e+06,1.528173e+06,475905.990000,295499.890000,29.359678,26.377488


In [14]:
"""檢視 label_table（Parquet）"""
if catalog.exists("label_table"):
    df = catalog.load("label_table")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))
    display(df.describe())

    # Label 分佈
    if "label" in df.columns:
        print(f"\nLabel 分佈:\n{df['label'].value_counts()}")
    if "prod_name" in df.columns:
        print(f"\nProd_name 分佈:\n{df['prod_name'].value_counts()}")
        # 各產品正樣本比例
        pos_rate = df.groupby("prod_name")["label"].mean().sort_values(ascending=False)
        print(f"\n各產品正樣本比例:\n{pos_rate}")
    if "snap_date" in df.columns:
        print(f"\nSnap_date 分佈:\n{df['snap_date'].value_counts().sort_index()}")
else:
    print("label_table 不存在，請先執行 dataset building pipeline。")

Shape: (3000, 7)

Dtypes:
snap_date           datetime64[ns]
cust_id                     object
cust_segment_typ            object
apply_start_date    datetime64[ns]
apply_end_date      datetime64[ns]
label                        int64
prod_name                   object
dtype: object

Null 統計:
snap_date           0
cust_id             0
cust_segment_typ    0
apply_start_date    0
apply_end_date      0
label               0
prod_name           0
dtype: int64


,snap_date,cust_id,cust_segment_typ,apply_start_date,apply_end_date,label,prod_name
0,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,fx
1,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,usd
2,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,stock
3,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,bond
4,2024-01-31,C000001,mass,2024-02-01,2024-03-01,1,mix
5,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,fx
6,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,usd
7,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,stock
8,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,bond
9,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,mix


,label
count,3000.000000
mean,0.099000
std,0.298712
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000



Label 分佈:
0    2703
1     297
Name: label, dtype: int64

Prod_name 分佈:
fx       600
usd      600
stock    600
bond     600
mix      600
Name: prod_name, dtype: int64

各產品正樣本比例:
prod_name
mix      0.111667
usd      0.108333
bond     0.096667
fx       0.093333
stock    0.085000
Name: label, dtype: float64

Snap_date 分佈:
2024-01-31    1000
2024-02-29    1000
2024-03-31    1000
Name: snap_date, dtype: int64


In [15]:
"""檢視 category_mappings（JSON）"""
if catalog.exists("category_mappings"):
    mappings = catalog.load("category_mappings")
    print(f"Type: {type(mappings)}")
    print(f"\n內容：")
    print(mappings)
else:
    print("category_mappings 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

內容：
{'prod_name': ['bond', 'fx', 'mix', 'stock', 'usd']}


In [ ]:
"""檢視 preprocessor（Pickle）"""
if catalog.exists("preprocessor"):
    preprocessor = catalog.load("preprocessor")
    print(f"Type: {type(preprocessor)}")
    if isinstance(preprocessor, dict):
        print(f"\nKeys: {list(preprocessor.keys())}")
        for k, v in preprocessor.items():
            print(f"\n--- {k} ---")
            print(f"Type: {type(v)}")
            print(v)
    else:
        if hasattr(preprocessor, "get_params"):
            print(f"\nParams:\n{preprocessor.get_params()}")

else:
    print("preprocessor 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

Keys: ['feature_columns', 'categorical_columns', 'category_mappings', 'drop_columns']

--- feature_columns ---
Type: <class 'list'>
['prod_name', 'total_aum', 'fund_aum', 'in_amt_sum_l1m', 'out_amt_sum_l1m', 'in_amt_ratio_l1m', 'out_amt_ratio_l1m']

--- categorical_columns ---
Type: <class 'list'>
['prod_name']

--- category_mappings ---
Type: <class 'dict'>
{'prod_name': ['bond', 'fx', 'mix', 'stock', 'usd']}

--- drop_columns ---
Type: <class 'list'>
['snap_date', 'cust_id', 'label', 'apply_start_date', 'apply_end_date', 'cust_segment_typ']


In [17]:
"""檢視 model（Pickle）"""
if catalog.exists("model"):
    model = catalog.load("model")
    print(f"Type: {type(model)}")
    if hasattr(model, "get_params"):
        print(f"\nParams:\n{model.get_params()}")

    # LightGBM feature importance
    if hasattr(model, "feature_importances_") and hasattr(model, "feature_name_"):
        import pandas as pd
        importance = pd.DataFrame({
            "feature": model.feature_name_,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)
        print(f"\nTop 20 Feature Importance:")
        display(importance.head(20))
else:
    print("model 不存在，請先執行 training pipeline。")

model 不存在，請先執行 training pipeline。
